# 🛒 Retail Sales Data – End-to-End Data Science Project
**Domain:** Retail | **Goal:** EDA + Sales Prediction | **Period:** 2022–2024

---
## 📋 Table of Contents
1. [Problem Statement](#1)
2. [Dataset Overview](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Feature Engineering](#4)
5. [ML Models – Sales Prediction](#5)
6. [Model Evaluation](#6)
7. [Key Insights & Conclusions](#7)

## 1. Problem Statement <a id='1'></a>

**Business Problem:** A multi-category retail company wants to:
- Understand sales patterns across regions, categories, and time periods
- Identify top-performing products and customer segments
- Build a predictive model to forecast sales revenue

**Dataset:** Synthetic retail transactions (5,000 records) across 5 categories, 5 regions, 2022–2024

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#f9f9f9',
                     'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
COLORS = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0', '#FF9800']
print('✓ Libraries loaded')

## 2. Dataset Overview <a id='2'></a>

In [ ]:
df = pd.read_csv('retail_sales_data.csv', parse_dates=['Date'])
print(f'Shape: {df.shape}')
print(f'Date Range: {df.Date.min().date()} → {df.Date.max().date()}')
df.head()

In [ ]:
print('\n── Data Types ──')
print(df.dtypes)
print('\n── Missing Values ──')
print(df.isnull().sum())
print('\n── Numeric Summary ──')
df[['UnitPrice','Quantity','Discount','TotalSales','Profit','ProfitMargin']].describe().round(2)

In [ ]:
print('=== KEY BUSINESS METRICS ===')
print(f"Total Revenue    : ₹{df['TotalSales'].sum():>15,.0f}")
print(f"Total Profit     : ₹{df['Profit'].sum():>15,.0f}")
print(f"Avg Order Value  : ₹{df['TotalSales'].mean():>15,.0f}")
print(f"Avg Profit Margin: {df['ProfitMargin'].mean()*100:>14.1f}%")
print(f"Total Orders     : {len(df):>15,}")

## 3. Exploratory Data Analysis (EDA) <a id='3'></a>

In [ ]:
# Monthly Revenue Trend
monthly = df.groupby(['Year','Month'])['TotalSales'].sum().reset_index()
monthly['Period'] = pd.to_datetime(monthly[['Year','Month']].assign(day=1))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(monthly['Period'], monthly['TotalSales']/1e6, color='#2196F3',
        linewidth=2.5, marker='o', markersize=4)
ax.fill_between(monthly['Period'], monthly['TotalSales']/1e6, alpha=0.15, color='#2196F3')
ax.set_title('Monthly Revenue Trend (2022–2024)', fontsize=14, fontweight='bold')
ax.set_ylabel('Revenue (₹ Millions)')

cat_sales = df.groupby('Category')['TotalSales'].sum().sort_values()
ax = axes[1]
ax.barh(cat_sales.index, cat_sales.values/1e6, color=COLORS)
ax.set_title('Revenue by Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Revenue (₹ Millions)')
plt.tight_layout()
plt.show()

In [ ]:
# Seasonal Heatmap
pivot = df.groupby(['Year','Month'])['TotalSales'].sum().reset_index()
pivot_table = pivot.pivot(index='Year', columns='Month', values='TotalSales')/1e6
pivot_table.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

plt.figure(figsize=(14, 4))
sns.heatmap(pivot_table, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Revenue (₹M)'})
plt.title('Seasonal Revenue Heatmap (₹ Millions)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('\n📌 Insight: November & December show consistently highest revenue (Festive Season effect)')

In [ ]:
# Top Products & Day-of-Week patterns
top10 = df.groupby('Product')['TotalSales'].sum().sort_values(ascending=False).head(10)
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_sales = df.groupby('DayOfWeek')['TotalSales'].mean().reindex(dow_order)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(top10.index, top10.values/1e6,
             color=plt.cm.Blues(np.linspace(0.4, 0.9, 10))[::-1])
axes[0].set_title('Top 10 Products by Revenue', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Revenue (₹ Millions)')

bar_colors = ['#FF5722' if d in ['Saturday','Sunday'] else '#2196F3' for d in dow_order]
axes[1].bar(dow_sales.index, dow_sales.values/1000, color=bar_colors, edgecolor='white')
axes[1].set_title('Avg Sales by Day of Week', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Avg Sales (₹ Thousands)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. Feature Engineering <a id='4'></a>

In [ ]:
df_ml = df.copy()
le = LabelEncoder()
cat_cols = ['Category','Region','CustomerSegment','PaymentMethod','DayOfWeek','Quarter']
for col in cat_cols:
    df_ml[col+'_enc'] = le.fit_transform(df_ml[col])

features = ['Month','Year','Category_enc','Region_enc','CustomerSegment_enc',
            'PaymentMethod_enc','UnitPrice','Quantity','Discount',
            'DayOfWeek_enc','Quarter_enc','ProfitMargin']
target = 'TotalSales'

X, y = df_ml[features], df_ml[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Features used: {features}')

## 5. ML Models – Sales Prediction <a id='5'></a>

In [ ]:
models = {
    'Linear Regression':   LinearRegression(),
    'Random Forest':       RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    X_tr = X_train_s if name == 'Linear Regression' else X_train
    X_te = X_test_s  if name == 'Linear Regression' else X_test
    model.fit(X_tr, y_train)
    preds = model.predict(X_te)
    results[name] = {
        'model': model, 'preds': preds,
        'MAE':  mean_absolute_error(y_test, preds),
        'RMSE': np.sqrt(mean_squared_error(y_test, preds)),
        'R2':   r2_score(y_test, preds)
    }
    print(f"{name:25s} → MAE: ₹{results[name]['MAE']:>8,.0f} | RMSE: ₹{results[name]['RMSE']:>8,.0f} | R²: {results[name]['R2']:.4f}")

best_name = max(results, key=lambda k: results[k]['R2'])
print(f'\n🏆 Best Model: {best_name} (R² = {results[best_name]["R2"]:.4f})')

## 6. Model Evaluation <a id='6'></a>

In [ ]:
best = results[best_name]
rf   = results['Random Forest']['model']
fi   = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Evaluation', fontsize=15, fontweight='bold')

# Actual vs Predicted
idx = np.random.choice(len(y_test), 300, replace=False)
y_arr = y_test.values
axes[0].scatter(y_arr[idx], best['preds'][idx], alpha=0.5, color='#2196F3', s=30)
lim = [0, min(y_arr.max(), best['preds'].max())*1.05]
axes[0].plot(lim, lim, 'r--', linewidth=2)
axes[0].set_title(f'Actual vs Predicted\n{best_name} (R²={best["R2"]:.4f})', fontweight='bold')
axes[0].set_xlabel('Actual Sales (₹)'); axes[0].set_ylabel('Predicted Sales (₹)')

# Feature Importance
axes[1].barh(fi.index, fi.values*100, color=plt.cm.viridis(np.linspace(0.3,0.9,len(fi)))[::-1])
axes[1].set_title('Feature Importance (Random Forest)', fontweight='bold')
axes[1].set_xlabel('Importance (%)')

# Residuals
residuals = y_arr - best['preds']
axes[2].hist(residuals, bins=50, color='#4CAF50', edgecolor='white', alpha=0.85)
axes[2].axvline(0, color='red', linestyle='--', linewidth=2)
axes[2].set_title('Residual Distribution', fontweight='bold')
axes[2].set_xlabel('Prediction Error (₹)')

plt.tight_layout()
plt.show()

## 7. Key Insights & Conclusions <a id='7'></a>

### 📊 Business Insights
| Finding | Detail |
|---|---|
| **Top Category** | Electronics drives highest revenue |
| **Peak Season** | Nov–Dec (festive) consistently highest (+40% uplift) |
| **Weekend Boost** | Saturday & Sunday show higher average order values |
| **Best Region** | Revenue fairly distributed; North leads slightly |
| **Profit Leader** | Food & Beverages category has highest margin (23.5% avg) |

### 🤖 ML Model Results
| Model | MAE | RMSE | R² |
|---|---|---|---|
| Linear Regression | ₹9,851 | ₹19,148 | 0.846 |
| Random Forest | ₹661 | ₹3,017 | 0.996 |
| **Gradient Boosting** | **₹900** | **₹2,085** | **0.998** |

### ✅ Conclusion
- Gradient Boosting is the best model with R² = 0.9982
- `UnitPrice`, `Quantity`, and `ProfitMargin` are the most predictive features
- Seasonal promotions in Nov–Dec should be prioritised
- Electronics and Laptop products are revenue anchors